# Querying TEI with XPath

**Computational Text Analysis Pack · Module 20 — Encoding as Method**

*Technique:* XPath over encoded text; structural queries a plain-text search cannot make

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/09-encoding-as-method/20b_querying_tei_with_xpath.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Encoding earns its cost at query time. Once names, dates and structure are
marked, you can ask questions that are impossible over plain text — *which
people are mentioned in the same paragraph as a date before 1850* — and get an
answer that is exact rather than approximate.

**XPath** is the query language for that, and it is the third query language
this course has now covered, after regex (characters) and CQL (tokens). XPath
works over **structure**.

## The tooling

`xml.etree.ElementTree` supports a useful XPath subset and is standard library.
**lxml** supports full XPath 1.0 including `following-sibling`, `ancestor` and
positional predicates. **XQuery** and **XSLT** go further — XQuery for querying
collections, XSLT for transforming TEI into HTML or LaTeX. **TEI Publisher**
and **BaseX** are the usual infrastructure for a real encoded corpus.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
import xml.etree.ElementTree as ET
import re
from xml.sax.saxutils import escape

TEI_NS = "http://www.tei-c.org/ns/1.0"
ns = {"t": TEI_NS}

def encode(text, title, author):
    body = escape(str(text))
    body = re.sub(r"\b((?:Mr|Mrs|Dr|Sir|Lord|Captain|Chief|King)\.?\s+[A-Z]\w+)",
                  r"<persName>\1</persName>", body)
    body = re.sub(r"\b(1[6-9]\d{2})\b", r'<date when="\1">\1</date>', body)
    body = re.sub(r"\b(?:at|in|near)\s+([A-Z][a-z]{3,})",
                  r"<placeName>\1</placeName>", body)
    paras = "".join(f"<p>{p.strip()}</p>"
                    for p in re.split(r"(?<=\.)\s{2,}", body) if p.strip())
    return (f'<TEI xmlns="{TEI_NS}"><teiHeader><fileDesc><titleStmt>'
            f'<title>{escape(str(title))}</title><author>{escape(str(author))}</author>'
            f'</titleStmt></fileDesc></teiHeader><text><body>{paras}</body></text></TEI>')

# A small encoded collection, which is what a TEI project actually looks like.
collection = []
for _, row in df_colonial.head(300).iterrows():
    try:
        collection.append(ET.fromstring(encode(row["text"], row["title"], row["author"])))
    except ET.ParseError:
        continue
print(f"{len(collection)} encoded documents in the collection")

In [ ]:
# XPath 1: every person named anywhere in the collection.
people = Counter()
for doc in collection:
    for el in doc.findall(".//t:persName", ns):
        if el.text:
            people[" ".join(el.text.split())] += 1
print(f"{len(people)} distinct persName elements\n")
for name, n in people.most_common(10):
    print(f"   {name:<28}{n:>4}")

## The query plain text cannot answer

*Find every person named in a paragraph that also contains a date before 1850.*
Over plain text this is a proximity heuristic with an arbitrary window. Over TEI
it is a structural fact: same `<p>` element, and the `@when` attribute is a
number you can compare.

In [ ]:
hits = []
for doc in collection:
    title = doc.findtext(".//t:title", default="?", namespaces=ns)
    for para in doc.findall(".//t:p", ns):
        dates = [int(d.get("when")) for d in para.findall("t:date", ns)
                 if d.get("when", "").isdigit()]
        names = [n.text for n in para.findall("t:persName", ns) if n.text]
        early = [d for d in dates if d < 1850]
        if early and names:
            hits.append((title[:34], min(early), names))

print(f"{len(hits)} paragraphs pair a pre-1850 date with a named person\n")
for title, year, names in hits[:8]:
    print(f"   {year}  {title:<36}{', '.join(n.strip() for n in names[:3])}")
print()
print("The co-occurrence is guaranteed by the markup, not inferred from a")
print("character window. That is what the encoding bought.")

In [ ]:
# XPath 2: the shape of the collection itself.
print(f"{'metric':<34}{'value':>10}")
print("-" * 44)
print(f"{'documents':<34}{len(collection):>10,}")
print(f"{'paragraphs':<34}{sum(len(d.findall('.//t:p', ns)) for d in collection):>10,}")
print(f"{'persName elements':<34}{sum(len(d.findall('.//t:persName', ns)) for d in collection):>10,}")
print(f"{'date elements':<34}{sum(len(d.findall('.//t:date', ns)) for d in collection):>10,}")
print(f"{'placeName elements':<34}{sum(len(d.findall('.//t:placeName', ns)) for d in collection):>10,}")

years = Counter()
for doc in collection:
    for d in doc.findall(".//t:date", ns):
        w = d.get("when", "")
        if w.isdigit():
            years[int(w) // 10 * 10] += 1
if years:
    xs = sorted(years)
    plt.figure(figsize=(7.5, 3.6))
    plt.bar([str(x) for x in xs], [years[x] for x in xs], color="steelblue")
    plt.xticks(rotation=45, fontsize=8)
    plt.ylabel("<date> elements")
    plt.title("Which decades the corpus talks about")
    plt.tight_layout()

## The limit you must not forget

Every number above is a measurement of **our encoding**, not of the corpus. The
regex tagged fewer than half the real names and almost none of the non-English
ones. A structural query gives an exact answer to the question "what does this
markup say", and that is a different question from "what does this text
contain".

In [ ]:
all_caps = Counter()
for _, row in df_colonial.head(300).iterrows():
    all_caps.update(re.findall(r"\b[A-Z][a-z]{3,}\b", str(row["text"])))
tagged = set(w for name in people for w in name.split())
untagged = [(w, n) for w, n in all_caps.most_common(300) if w not in tagged]

print(f"capitalised word types in the source : {len(all_caps):,}")
print(f"word types inside a <persName>       : {len(tagged):,}")
print(f"\nFrequent capitalised words our encoding never tagged:")
for w, n in untagged[:12]:
    print(f"   {w:<18}{n:>5}")
print()
print("Some are places, some are months, some are people the rule missed. An")
print("edition would resolve each by hand. Report the recall of your encoding,")
print("or a reader will mistake its silence for the text's.")

### 🔧 Your turn 1

Write a query for paragraphs containing a placeName but no persName. What kind of writing are they?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

They skew heavily toward description of terrain and route — the passages where
the landscape is the subject and nobody is present. That contrast between
peopled and unpeopled description is a genuinely literary observation, and it is
reachable only because the markup distinguishes the two entity types. It is also
a good illustration of why encoding precedes analysis: no amount of clever
counting recovers a distinction the data never recorded.

</details>